In [ ]:
from os import getenv

from dotenv import load_dotenv

from classifai.indexers import VectorStore
from classifai.indexers.hooks.default_hooks import HuggingFaceRagHook
from classifai.vectorisers import HuggingFaceVectoriser

load_dotenv()
API_KEY = getenv("HF_API_KEY")

CONTEXT_PROMPT_CLASSIFICATION = """You are an agent helping classify job descriptions.
Your task is to identify the most relevant retrieved document for a given query.
Make your description based on the 'doc_text' column, comparing each option to the 'query_text'
column.
You must identify the row with the most relevant retrieved document.
When you have identified the most relevant row, respond strictly in the format described in the
Output Format section.
"""

RESPONSE_PROMPT_CLASSIFICATION = """
Respond ONLY with a single JSON string, which is a list of boolean values the same length as the
DataFrame you received - in the JSON list, put a "1" at the element corresponding to the most
relevant row, and "0"s for all other elements.
Make sure to avoid backticks.
Take the format of the example below as a strict guide.
Example (in the case where there are 5 rows, and the second is the most relevant):
{"answer":[0,1,0,0,0]}
"""

classification_hf_rag_hook = HuggingFaceRagHook(
    context_prompt=CONTEXT_PROMPT_CLASSIFICATION,
    response_prompt=RESPONSE_PROMPT_CLASSIFICATION,
    api_key=API_KEY,
    model_name="google/gemma-4-31B-it",
    model_provider="novita",
)

vectoriser = HuggingFaceVectoriser(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = VectorStore(
    file_name="./data/fake_soc_dataset.csv",
    data_type="csv",
    vectoriser=vectoriser,
    output_dir="./demo_vdb",
    overwrite=True,
    hooks={"search_postprocess": classification_hf_rag_hook},
)

In [ ]:
from classifai.indexers.dataclasses import VectorStoreSearchInput

input_data = VectorStoreSearchInput({"id": [1], "query": ["a pizza maker and barista working for Flout"]})

In [ ]:
vectorstore.search(input_data)